# ✈️ Airline Loyalty Behavioral Intelligence — Exploratory Data Analysis

**Dataset:** Canadian Airline Loyalty Program (2017–2018)  
**Customers:** 16,737 loyalty members  
**Goal:** Understand customer behavior patterns to build a churn prediction model

---
## Contents
1. [Dataset Overview](#1-dataset-overview)
2. [Customer Demographics](#2-customer-demographics)
3. [Churn Rate Analysis](#3-churn-rate-analysis)
4. [Flight Behavior Analysis](#4-flight-behavior-analysis)
5. [CLV & Revenue Analysis](#5-clv--revenue-analysis)
6. [Points & Engagement Analysis](#6-points--engagement-analysis)
7. [Feature Correlations](#7-feature-correlations)
8. [Key Findings](#8-key-findings)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (14, 5), 'font.size': 11})

# ── Load data ────────────────────────────────────────────────────────────────
loyalty  = pd.read_csv('../data/raw/Customer Loyalty History.csv')
activity = pd.read_csv('../data/raw/Customer Flight Activity.csv')

# Standardise column names
loyalty.columns  = [c.strip().replace(' ', '_').lower() for c in loyalty.columns]
activity.columns = [c.strip().replace(' ', '_').lower() for c in activity.columns]

# Build activity date
activity['date'] = pd.to_datetime({'year': activity['year'], 'month': activity['month'], 'day': 1})

# Churn label: churned if cancelled OR no flights in final 12 months
PREDICTION_DATE   = pd.Timestamp('2017-12-31')
cutoff_12m        = PREDICTION_DATE - pd.DateOffset(months=12)
last_flight       = activity.groupby('loyalty_number')['date'].max()
activity_churn    = (last_flight < cutoff_12m).astype(int)
hard_churn        = loyalty['cancellation_year'].notna().astype(int)

loyalty['activity_churn'] = loyalty['loyalty_number'].map(activity_churn).fillna(1)
loyalty['hard_churn']     = hard_churn
loyalty['churn']          = ((loyalty['activity_churn'] == 1) | (loyalty['hard_churn'] == 1)).astype(int)

print(f'Loyalty dataset:  {loyalty.shape}')
print(f'Activity dataset: {activity.shape}')
print(f'Overall churn rate: {loyalty["churn"].mean()*100:.2f}%')

---
## 1. Dataset Overview

In [ ]:
print('='*50)
print('LOYALTY DATASET')
print('='*50)
print(loyalty.dtypes.to_string())
print(f'\nMissing values:\n{loyalty.isnull().sum()[loyalty.isnull().sum() > 0]}')

In [ ]:
print('='*50)
print('ACTIVITY DATASET')
print('='*50)
print(activity.dtypes.to_string())
print(f'\nDate range: {activity["date"].min().date()} → {activity["date"].max().date()}')
print(f'Records per customer (avg): {len(activity)/activity["loyalty_number"].nunique():.1f}')

In [ ]:
loyalty.describe().T[['mean','std','min','max']].round(2)

---
## 2. Customer Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gender distribution
gender_counts = loyalty['gender'].value_counts()
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#3498db', '#e74c3c'], startangle=90)
axes[0].set_title('Gender Distribution', fontweight='bold')

# Loyalty card tier
card_counts = loyalty['loyalty_card'].value_counts()
bars = axes[1].bar(card_counts.index, card_counts.values,
                   color=['#2ecc71', '#f39c12', '#e74c3c'], alpha=0.85)
for bar, val in zip(bars, card_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', fontsize=10)
axes[1].set_title('Loyalty Card Tier', fontweight='bold')
axes[1].set_ylabel('Customers')
axes[1].grid(True, alpha=0.3)

# Education level
edu_order = ['High School or Below', 'College', 'Bachelor', 'Master', 'Doctor']
edu_counts = loyalty['education'].value_counts().reindex(edu_order, fill_value=0)
axes[2].barh(edu_counts.index, edu_counts.values, color='#9b59b6', alpha=0.8)
axes[2].set_title('Education Level', fontweight='bold')
axes[2].set_xlabel('Customers')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Customer Demographics Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Enrollment year trend
enroll = loyalty['enrollment_year'].value_counts().sort_index()
axes[0].bar(enroll.index, enroll.values, color='#3498db', alpha=0.8)
axes[0].set_title('New Customer Enrollments by Year', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('New Customers')
axes[0].grid(True, alpha=0.3)

# Marital status
ms_counts = loyalty['marital_status'].value_counts()
axes[1].pie(ms_counts, labels=ms_counts.index, autopct='%1.1f%%',
            colors=['#1abc9c', '#e67e22', '#95a5a6'], startangle=90)
axes[1].set_title('Marital Status Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3. Churn Rate Analysis

In [ ]:
# Overall churn breakdown
total      = len(loyalty)
churned    = loyalty['churn'].sum()
retained   = total - churned
hard_only  = ((loyalty['hard_churn']==1) & (loyalty['activity_churn']==0)).sum()
act_only   = ((loyalty['hard_churn']==0) & (loyalty['activity_churn']==1)).sum()
both       = ((loyalty['hard_churn']==1) & (loyalty['activity_churn']==1)).sum()

print(f'Total customers  : {total:,}')
print(f'Churned          : {churned:,}  ({churned/total*100:.1f}%)')
print(f'Retained         : {retained:,}  ({retained/total*100:.1f}%)')
print()
print(f'Churn breakdown:')
print(f'  Hard churn only (cancelled)   : {hard_only:,}')
print(f'  Activity churn only (inactive): {act_only:,}')
print(f'  Both definitions              : {both:,}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn by loyalty card
card_churn = loyalty.groupby('loyalty_card')['churn'].agg(['mean','count']).reset_index()
card_churn['mean'] *= 100
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = axes[0].bar(card_churn['loyalty_card'], card_churn['mean'], color=colors, alpha=0.85)
for bar, val in zip(bars, card_churn['mean']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Churn Rate by Loyalty Card Tier', fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, card_churn['mean'].max() * 1.3)
axes[0].grid(True, alpha=0.3)

# Churn by gender
gender_churn = loyalty.groupby('gender')['churn'].mean() * 100
axes[1].bar(gender_churn.index, gender_churn.values, color=['#3498db','#e74c3c'], alpha=0.85)
for i, (idx, val) in enumerate(gender_churn.items()):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Churn Rate by Gender', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, gender_churn.max() * 1.3)
axes[1].grid(True, alpha=0.3)

# Churn by education
edu_order = ['High School or Below', 'College', 'Bachelor', 'Master', 'Doctor']
edu_churn = loyalty.groupby('education')['churn'].mean().reindex(edu_order) * 100
axes[2].barh(edu_churn.index, edu_churn.values, color='#9b59b6', alpha=0.8)
for i, val in enumerate(edu_churn.values):
    axes[2].text(val + 0.2, i, f'{val:.1f}%', va='center', fontweight='bold')
axes[2].set_title('Churn Rate by Education Level', fontweight='bold')
axes[2].set_xlabel('Churn Rate (%)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Churn Rate by Customer Segment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Flight Behavior Analysis

In [ ]:
# Aggregate activity per customer
cust_activity = activity.groupby('loyalty_number').agg(
    total_flights    = ('total_flights',     'sum'),
    total_distance   = ('distance',          'sum'),
    active_months    = ('date',              'nunique'),
    last_date        = ('date',              'max'),
).reset_index()
cust_activity['avg_flights_mo'] = cust_activity['total_flights'] / cust_activity['active_months'].clip(lower=1)
cust_activity['recency_months'] = ((PREDICTION_DATE - cust_activity['last_date']) / pd.Timedelta(days=30.44)).clip(lower=0)

# Merge churn label
cust_activity = cust_activity.merge(loyalty[['loyalty_number','churn']], on='loyalty_number', how='left')
cust_activity['churn'] = cust_activity['churn'].fillna(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Total flights distribution: churned vs retained
for label, grp in cust_activity.groupby('churn'):
    name = 'Churned' if label == 1 else 'Retained'
    color = '#e74c3c' if label == 1 else '#2ecc71'
    data = grp['total_flights'].clip(upper=grp['total_flights'].quantile(0.99))
    axes[0].hist(data, bins=40, alpha=0.5, label=name, color=color, density=True)
axes[0].set_title('Total Flights Distribution: Churned vs Retained', fontweight='bold')
axes[0].set_xlabel('Total Flights')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Recency distribution
for label, grp in cust_activity.groupby('churn'):
    name  = 'Churned' if label == 1 else 'Retained'
    color = '#e74c3c' if label == 1 else '#2ecc71'
    axes[1].hist(grp['recency_months'].clip(upper=24), bins=25, alpha=0.5,
                 label=name, color=color, density=True)
axes[1].set_title('Recency Distribution: Churned vs Retained', fontweight='bold')
axes[1].set_xlabel('Months Since Last Flight')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly flight trends across the dataset
monthly = activity.groupby('date')['total_flights'].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(monthly['date'], monthly['total_flights'], color='#3498db', linewidth=2)
axes[0].fill_between(monthly['date'], monthly['total_flights'], alpha=0.15, color='#3498db')
axes[0].set_title('Total Flights per Month Across All Customers', fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Total Flights')
axes[0].grid(True, alpha=0.3)

# Seasonality — average flights by month number
activity['month_name'] = activity['date'].dt.strftime('%b')
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
seasonal = activity.groupby(activity['date'].dt.month)['total_flights'].mean()
axes[1].bar(range(1,13), seasonal.values, color='#e67e22', alpha=0.8)
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(month_order)
axes[1].set_title('Average Flights by Month (Seasonality)', fontweight='bold')
axes[1].set_ylabel('Average Flights')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. CLV & Revenue Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# CLV distribution
clv_99 = loyalty['clv'].quantile(0.99)
axes[0].hist(loyalty['clv'].clip(upper=clv_99), bins=50, color='#27ae60', alpha=0.8)
axes[0].axvline(loyalty['clv'].median(), color='red', linestyle='--', label=f'Median: ${loyalty["clv"].median():,.0f}')
axes[0].set_title('CLV Distribution', fontweight='bold')
axes[0].set_xlabel('Customer Lifetime Value (CAD)')
axes[0].set_ylabel('Customers')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CLV by loyalty card
card_clv = loyalty.groupby('loyalty_card')['clv'].median()
bars = axes[1].bar(card_clv.index, card_clv.values, color=['#2ecc71','#f39c12','#e74c3c'], alpha=0.85)
for bar, val in zip(bars, card_clv.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'${val:,.0f}', ha='center', fontweight='bold')
axes[1].set_title('Median CLV by Loyalty Card Tier', fontweight='bold')
axes[1].set_ylabel('Median CLV (CAD)')
axes[1].grid(True, alpha=0.3)

# CLV: churned vs retained
for label, grp in loyalty.groupby('churn'):
    name  = 'Churned' if label == 1 else 'Retained'
    color = '#e74c3c' if label == 1 else '#2ecc71'
    axes[2].hist(grp['clv'].clip(upper=clv_99), bins=40, alpha=0.5, label=name, color=color, density=True)
axes[2].set_title('CLV Distribution: Churned vs Retained', fontweight='bold')
axes[2].set_xlabel('CLV (CAD)')
axes[2].set_ylabel('Density')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Customer Lifetime Value Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Overall median CLV : ${loyalty["clv"].median():,.2f}')
print(f'Churned median CLV : ${loyalty[loyalty["churn"]==1]["clv"].median():,.2f}')
print(f'Retained median CLV: ${loyalty[loyalty["churn"]==0]["clv"].median():,.2f}')

---
## 6. Points & Engagement Analysis

In [ ]:
# Points per customer
pts = activity.groupby('loyalty_number').agg(
    total_earned   = ('points_accumulated', 'sum'),
    total_redeemed = ('points_redeemed',    'sum'),
).reset_index()
pts['redemption_rate'] = (pts['total_redeemed'] / pts['total_earned'].clip(lower=1)).clip(0, 1)
pts['points_balance']  = (pts['total_earned'] - pts['total_redeemed']).clip(lower=0)
pts = pts.merge(loyalty[['loyalty_number','churn','loyalty_card']], on='loyalty_number', how='left')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Redemption rate distribution
for label, grp in pts.groupby('churn'):
    name  = 'Churned' if label == 1 else 'Retained'
    color = '#e74c3c' if label == 1 else '#2ecc71'
    axes[0].hist(grp['redemption_rate'].dropna(), bins=30, alpha=0.5,
                 label=name, color=color, density=True)
axes[0].set_title('Points Redemption Rate: Churned vs Retained', fontweight='bold')
axes[0].set_xlabel('Redemption Rate (0=never redeemed, 1=fully redeemed)')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Average redemption rate by card tier
tier_redemption = pts.groupby('loyalty_card')['redemption_rate'].mean() * 100
axes[1].bar(tier_redemption.index, tier_redemption.values,
            color=['#2ecc71','#f39c12','#e74c3c'], alpha=0.85)
for i, val in enumerate(tier_redemption.values):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Avg Redemption Rate by Card Tier', fontweight='bold')
axes[1].set_ylabel('Redemption Rate (%)')
axes[1].grid(True, alpha=0.3)

# Points balance distribution (log scale)
balance_99 = pts['points_balance'].quantile(0.99)
axes[2].hist(pts['points_balance'].clip(upper=balance_99), bins=40, color='#8e44ad', alpha=0.8)
axes[2].set_title('Points Balance Distribution', fontweight='bold')
axes[2].set_xlabel('Unredeemed Points Balance')
axes[2].set_ylabel('Customers')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Points & Engagement Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Avg redemption rate — Churned : {pts[pts["churn"]==1]["redemption_rate"].mean()*100:.1f}%')
print(f'Avg redemption rate — Retained: {pts[pts["churn"]==0]["redemption_rate"].mean()*100:.1f}%')

---
## 7. Feature Correlations

In [ ]:
# Build a compact feature frame for correlation analysis
feat = cust_activity[['loyalty_number','total_flights','avg_flights_mo','recency_months','active_months','churn']].copy()
feat = feat.merge(pts[['loyalty_number','redemption_rate','points_balance','total_earned']], on='loyalty_number', how='left')
feat = feat.merge(loyalty[['loyalty_number','clv','clv_log' if 'clv_log' in loyalty.columns else 'clv']].rename(columns={'clv':'clv'}), on='loyalty_number', how='left')
feat = feat.drop(columns=['loyalty_number']).fillna(0)

# Correlation with churn
corr_churn = feat.corr()['churn'].drop('churn').sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_churn.values]
axes[0].barh(corr_churn.index[::-1], corr_churn.values[::-1], color=colors[::-1], alpha=0.85)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Feature Correlations with Churn', fontweight='bold')
axes[0].set_xlabel('Pearson Correlation')
axes[0].grid(True, alpha=0.3)

sns.heatmap(feat.corr(), ax=axes[1], cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 9},
            square=True, linewidths=0.5)
axes[1].set_title('Full Correlation Matrix', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 8. Key Findings

| # | Finding | Business Implication |
|---|---------|---------------------|
| 1 | **Overall churn rate: ~16%** — combined (cancellation OR 12-month inactivity) | 1 in 6 customers is at risk |
| 2 | **Recency is the strongest predictor** — churned customers show much higher recency months | Early inactivity detection is key |
| 3 | **Star card tier churns most** — lowest CLV, least committed customers | Tier-specific retention strategies needed |
| 4 | **Churned customers have lower redemption rates** — disengaged from the points program | Redemption nudges could reduce churn |
| 5 | **CLV is lower for churned customers** — but not dramatically | Even low-CLV churners represent lost revenue at scale |
| 6 | **Clear seasonality** — flight peaks in summer (Jun–Aug) and holiday season (Nov–Dec) | Seasonal campaigns should target at-risk customers before these windows |
| 7 | **Active month ratio is a strong signal** — retained customers fly more consistently across months | Consistency > total volume for loyalty |


In [ ]:
# Summary statistics table
summary = pd.DataFrame({
    'Metric': [
        'Total Customers', 'Churned', 'Retained', 'Churn Rate',
        'Median CLV', 'Avg Flights/Month', 'Avg Redemption Rate',
        'Total Activity Records'
    ],
    'Overall': [
        f"{len(loyalty):,}",
        f"{loyalty['churn'].sum():,}",
        f"{(loyalty['churn']==0).sum():,}",
        f"{loyalty['churn'].mean()*100:.1f}%",
        f"${loyalty['clv'].median():,.0f}",
        f"{cust_activity['avg_flights_mo'].mean():.2f}",
        f"{pts['redemption_rate'].mean()*100:.1f}%",
        f"{len(activity):,}"
    ]
})
summary